Nearest-grid city forecasts (silver layer).

Snaps each city in `dim_city` to its nearest forecast grid point by:
1. Exploding unique grid locations into H3 k-ring neighbors (res 5, ~12.8 km).
2. Hash-joining cities to candidate grid cells (broadcast, no shuffle).
3. Picking the closest grid via exact `st_distancesphere` distance.

Writes results to `silver_city_hourly_forecast`.

In [0]:
from pyspark.sql import functions as F

RES = 5  # ~12.8 km guaranteed 1-ring radius (covers 3–6 km API grid snap)
K = 1

# 1. Explode the smaller unique grid-locations table (7x) and broadcast it
fc = spark.table("workspace.open_meteo.silver_hourly_forecast")
locs_c = (
    fc.select(
        F.col("latitude").alias("fc_lat"),
        F.col("longitude").alias("fc_lon"),
    )
    .distinct()
    .withColumn(
        "nbr",
        F.explode(F.expr(f"h3_kring(h3_longlatash3(fc_lon, fc_lat, {RES}), {K})")),
    )
)

# 2. Keep dim_city at 1x rows
city_c = (
    spark.table("workspace.open_meteo.dim_city")
    .withColumn("cell", F.expr(f"h3_longlatash3(longitude, latitude, {RES})"))
)

# 3. Broadcast hash join (zero shuffle on dim_city) + exact distance tie-breaker
nearest = (
    city_c.join(F.broadcast(locs_c), city_c.cell == locs_c.nbr, how="inner")
    .withColumn(
        "distance_km",
        F.expr(
            "st_distancesphere(st_point(longitude, latitude), st_point(fc_lon, fc_lat)) / 1000.0"
        ),
    )
    .groupBy("id", "name", "country", "country_id", "latitude", "longitude", "timezone")
    .agg(
        F.min_by(
            F.struct("distance_km", "fc_lat", "fc_lon"),
            "distance_km",
        ).alias("best")
    )
    .select(
        "name",
        "country",
        "timezone",
        F.col("latitude").alias("city_lat"),
        F.col("longitude").alias("city_lon"),
        "best.fc_lat",
        "best.fc_lon",
        "best.distance_km",
    )
)

(
    nearest.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.open_meteo.silver_city_grid_map")
)
